[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassermakram/tiny-llm-education/blob/main/stage1_b_word_experiment.ipynb)

# Stage 1b: Zero-Layer Transformer (Word-Level) 🗺️
## target: Dialect Subway Lines (Bigram Transition Graphs)

Now that we understand how character transition metrics work, let's step up to the **Word Level**.

In a word-level language model, the vocabulary is composed of whole words.
Because we are predicting word-to-word transitions, training a full PyTorch model can take much longer due to the large vocabulary size. 

Since we already proved in Stage 1a that training a Zero-Layer network converges to the statistical bigram counts, we will **skip the training loop** and directly calculate the optimal matrices using direct counts!

### 💡 What we are showing:
We will map out the "subway lines" of the dialects. By filtering for the strongest transitions of core grammatical words (like 'الذي' in MSA vs 'اللي' in Masri), we can visualize the structural paths of each dialect as a directed graph.


In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q plotly pandas datasets networkx


In [ ]:
# 📦 Fetching MSA and Masri text (Words)
from datasets import load_dataset
import re

print("Fetching MSA from Wikipedia and Masri from Tweets...")

# Stream Wikipedia for MSA
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)

# Load Arabic Tweets for real street Masri
tweets_ds = load_dataset("amgadhasan/arabic_tweets_dialects", split="train")
# Filter for Egyptian tweets only
eg_tweets = tweets_ds.filter(lambda x: x['dialect'] == 'EG')

def clean_msa(stream, max_chars=500000):
    text = ""
    for article in stream:
        cleaned = re.sub(r'\s+', ' ', article['text'])
        cleaned = re.sub(r'[^\s\u0621-\u064A]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

def clean_masri(tweets, max_chars=500000):
    text = ""
    for t in tweets:
        cleaned = re.sub(r'\s+', ' ', t['text'])
        # Remove english mentions/links
        cleaned = re.sub(r'[a-zA-Z0-9_@]+', '', cleaned)
        cleaned = re.sub(r'[^\s\u0621-\u064A]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

print("Collecting MSA...")
msa_text = clean_msa(msa_stream)
print("Collecting Masri...")
masri_text = clean_masri(eg_tweets)

# Tokenise into words. At the word level our "tokens" are whitespace-separated
# words, so we split the cleaned text into a list of words for bigram counting.
# Shape: msa_words/masri_words are 1-D lists of word strings -> [n_words]
msa_words = msa_text.split()
masri_words = masri_text.split()

print(f"Loaded {len(msa_text)} chars / {len(msa_words)} words of MSA "
      f"and {len(masri_text)} chars / {len(masri_words)} words of Masri.")


In [ ]:
# 🧮 Compute Bigram Counts & Transition Probabilities Directly
# TODO: Calculate bigram transitions and normalise them into probabilities.
# Return a dict mapping (w1, w2) -> probability.
def compute_bigram_transitions(words, top_n=1000):
    pass

# msa_probs, msa_vocab = compute_bigram_transitions(msa_words)
# masri_probs, masri_vocab = compute_bigram_transitions(masri_words)


## 🗺️ Building the Dialect Subway Map
We'll select key "seed" words that capture grammar or common starting points (like 'في', 'من', 'اللي', 'الذي', 'هو', 'هي') and trace the most probable next-word paths to draw a subway map of the language!


In [ ]:
# 🕸️ Plot the Weighted-Bet Transition Trees (MSA vs Masri)
# TODO: For each dialect, grow a small tree from a seed word and draw it so the
# probability of each next word is obvious at a glance.
#
# Build three helpers and a driver:
#   1. build_transition_tree(probs, seed, max_depth=2, top_k=3)
#        BFS from `seed`; at each node keep the top_k likeliest NEW next words.
#        Return edges as (parent, child, prob, depth). Use top_k to select —
#        do NOT use a probability threshold (word-level distributions are flat,
#        so a threshold like 0.1 leaves the Masri tree empty).
#   2. tree_layout(edges, seed)
#        Deterministic left->right positions: x = depth, y = evenly spaced slot.
#        (Avoid nx.spring_layout — it produces an unreadable hairball.)
#   3. plot_weighted_tree(fig, row, edges, pos, color, seed)
#        Draw into a plotly subplot. Encode probability THREE ways:
#          • a directed arrow (annotation, parent -> child),
#          • arrow thickness scaled to the probability,
#          • the probability printed as a label on the arrow.
#
# Then build both trees (MSA seed 'الذي', Masri seed 'اللي') into a
# make_subplots(rows=2, cols=1) figure and call fig.show().